In [1]:
#  Demand Estimation - Logit Model

In [18]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS

In [19]:
loc= "PartD.csv"
#upload data;select variables of interest
data_raw = pd.read_csv(loc,sep=',',encoding='cp1252')
dt = data_raw.loc[:,('regionid','year', 'enrol_tot_num','enrol_lis_num','pdp', 'Part_D_Total_Premium','basic','Part_D_Drug_Deductible','unrestricted_drug','in_area_flag','Parent_Organization','Part_D_Basic_Premium')]
dt
# dt.describe()  
# dt[['regionid','year']].describe()
#note here there is a problem with missing variables
dt['enrol_lis_num'].isna().sum()
dt=dt.loc[dt['enrol_lis_num'].isna()==0,:] #excluding all observations with missing values

/var/folders/rt/71kk6f9n1mq72_4c0dxrjzbw0000gn/T/ipykernel_5386/3669128227.py:3: DtypeWarning: Columns (19,76) have mixed types. Specify dtype option on import or set low_memory=False.
  data_raw = pd.read_csv(loc,sep=',',encoding='cp1252')


In [20]:
#or use groupby
dt.groupby(['regionid','year']).enrol_tot_num.sum()
dt.groupby(['regionid','year','pdp']).enrol_tot_num.sum()

regionid  year  pdp
1         2006  0        2156.000000
                1      197030.000000
          2007  0        3436.196725
                1      210421.000000
          2008  0        6519.921049
                           ...      
34        2010  0           0.036561
                1       23735.000000
          2011  1       25200.000000
          2012  1        4388.000000
          2013  1        5712.000000
Name: enrol_tot_num, Length: 540, dtype: float64

In [29]:
dt

,regionid,year,enrol_tot_num,enrol_lis_num,pdp,Part_D_Total_Premium,basic,Part_D_Drug_Deductible,unrestricted_drug,in_area_flag,Parent_Organization,Part_D_Basic_Premium,mkt_id,mkt_id2,demand
0,25,2013,1768.0,90.0,0,0.00,0,0,184.0,70.0,Humana Inc.,0.00,252013,252013,1678.0
1,33,2013,1168.0,227.0,0,0.00,0,0,184.0,126.0,Humana Inc.,0.00,332013,332013,941.0
2,25,2013,139.0,28.0,0,19.00,0,0,184.0,12.0,Humana Inc.,19.00,252013,252013,111.0
3,25,2013,438.0,81.0,0,0.00,0,0,184.0,79.0,Humana Inc.,0.00,252013,252013,357.0
4,15,2013,4594.0,426.0,0,0.00,0,0,726.0,336.0,"UnitedHealth Group, Inc.",0.00,152013,152013,4168.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35138,11,2007,512.0,130.0,1,24.90,1,265,NaN,NaN,"America's Health Choice Medical Plans, Inc",24.90,112007,112007,382.0
35139,11,2006,778.0,164.0,1,48.44,1,250,48.0,3733.0,"America's Health Choice Medical Plans, Inc",48.44,112006,112006,614.0
35140,3,2007,51.0,0.0,1,44.70,0,0,NaN,NaN,"Capital District Physicians' Health Plan, Inc.",26.80,32007,32007,51.0
35141,3,2007,17.0,0.0,1,28.10,0,0,NaN,NaN,"Capital District Physicians' Health Plan, Inc.",27.00,32007,32007,17.0


In [39]:
dt.columns

Index(['regionid', 'year', 'enrol_tot_num', 'enrol_lis_num', 'pdp',
       'Part_D_Total_Premium', 'basic', 'Part_D_Drug_Deductible',
       'unrestricted_drug', 'in_area_flag', 'Parent_Organization',
       'Part_D_Basic_Premium', 'mkt_id', 'mkt_id2', 'demand'],
      dtype='object')

In [45]:
dt.describe()

,regionid,year,enrol_tot_num,enrol_lis_num,pdp,Part_D_Total_Premium,basic,Part_D_Drug_Deductible,unrestricted_drug,in_area_flag,Part_D_Basic_Premium,mkt_id2,demand
count,32155.000000,32155.000000,32155.000000,32155.000000,32155.000000,32155.000000,32155.000000,32155.000000,26605.000000,26551.00000,32098.000000,32155.000000,32155.000000
mean,16.515410,2009.134225,4655.586068,1829.087451,0.325268,28.467075,0.314446,74.633214,419.517158,1696.71809,22.436624,167163.231566,2826.498616
std,9.626083,2.027756,14948.773928,8116.245318,0.468482,22.796109,0.464302,120.812323,127.902404,3370.04206,17.088946,96260.894898,10005.110468
min,1.000000,2006.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,0.00000,-30.000000,12006.000000,0.000000
25%,9.000000,2008.000000,97.000000,13.000000,0.000000,13.500000,0.000000,0.000000,335.000000,286.00000,9.000000,92006.000000,52.000000
50%,15.000000,2009.000000,637.000000,105.000000,0.000000,27.000000,0.000000,0.000000,409.000000,791.00000,23.500000,152013.000000,363.000000
75%,25.000000,2011.000000,2945.500000,553.791105,1.000000,38.700000,1.000000,150.000000,503.000000,2010.50000,32.500000,252008.000000,1758.000000
max,34.000000,2013.000000,340276.000000,290104.000000,1.000000,361.800000,1.000000,325.000000,992.000000,46421.00000,119.699997,342013.000000,294531.000000


In [ ]:
#market shares

In [24]:
dt['mkt_id']= dt['regionid'].apply(str)+dt['year'].apply(str)
#dt['mkt_id2']= (dt['regionid']*10000)+dt['year'] #define market id region-year
dt['demand'] = dt.enrol_tot_num - dt.enrol_lis_num #calculate s_numerator (market share numerator for all j (j=0 included) in all R markets and add to data table

s0 = dt.loc[dt['pdp']==0].groupby('mkt_id').sum().demand #select "outside good" plans - demand for outside good plans
s = dt.groupby('mkt_id').sum().demand  #total demand in each market

mj= dt.loc[dt['pdp']==1].join(s, on='mkt_id',rsuffix='_total').join(s0,on='mkt_id',rsuffix='_0') #select plans pdp==1 (inside goods) add column with corresponding market total and outside good total
m= mj.assign(mkt_sh_j= lambda x: x['demand']/x['demand_total']).assign(mkt_sh_0= lambda x: x['demand_0']/x['demand_total'])
m

In [43]:
# =============================================================================
# #OLS regression
# =============================================================================
#defining y, x
y0 = np.log(m['mkt_sh_j'])-np.log(m['mkt_sh_0'] ) # to avoid divide by zero warning 
y = y0.mask(np.isinf(y0)| np.isnan(y0)) #ignore inf or nan
y.name='y'
x = m.loc[:,('Part_D_Total_Premium','basic','Part_D_Drug_Deductible','unrestricted_drug','in_area_flag')]
dummies = m.iloc[:,12:362]
#x=x.join([dummies_region,dummies_year])
x=sm.add_constant(x)

model = sm.OLS(y,x, missing='drop').fit() #model
print(model.summary()) #results 

with open('1.logit_OLS_regression_results.tex', 'w') as f:
    f.write(model.summary().as_latex())


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.101
Model:                            OLS   Adj. R-squared:                  0.100
Method:                 Least Squares   F-statistic:                     205.9
Date:                Sun, 22 Dec 2024   Prob (F-statistic):          9.44e-209
Time:                        15:35:30   Log-Likelihood:                -19244.
No. Observations:                9180   AIC:                         3.850e+04
Df Residuals:                    9174   BIC:                         3.854e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -2

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [48]:
# =============================================================================
#2SLS -  Hausman instrument
# =============================================================================

#Hausman IV
IV=dt.loc[dt['pdp']==1,('Part_D_Total_Premium','Parent_Organization','basic','regionid','year')]
#elements to exclude from the sum - own market values
temp1=IV.groupby(['Parent_Organization','basic','regionid','year']).sum()
temp2=IV.groupby(['Parent_Organization','basic','regionid','year']).count()
IV=IV.join(temp1,on=['Parent_Organization','basic','regionid','year'],rsuffix='_mktsum')
IV=IV.join(temp2,on=['Parent_Organization','basic','regionid','year'],rsuffix='_mktcount')

#totals
temp3=IV.groupby(['Parent_Organization','basic','year']).sum().Part_D_Total_Premium
temp4=IV.groupby(['Parent_Organization','basic','year']).count().Part_D_Total_Premium
IV=IV.join(temp3,on=['Parent_Organization','basic','year'],rsuffix='_allsum')
IV=IV.join(temp4,on=['Parent_Organization','basic','year'],rsuffix='_allcount')


ivhaus=(IV.Part_D_Total_Premium_allsum-IV.Part_D_Total_Premium_mktsum).div(IV.Part_D_Total_Premium_allcount-IV.Part_D_Total_Premium_mktcount) #some parent organization only have one market
ivhaus.name='ivhaus'
#IV=IV.assign(ivhaus= lambda x: (x['Part_D_Total_Premium_allsum']-x['Part_D_Total_Premium_mktsum'])/(x['Part_D_Total_Premium_allcount']-x['Part_D_Total_Premium_mktcount']))


#This gives an idea of what 2sls does, but standard errors are not properly dealt with
#First Stage
m2=x.join(ivhaus).join(y)
m2=m2.dropna() #!! Dropped missing values!!


#This gives an idea of what 2sls does, but standard errors are not properly dealt with
#First Stage
model1s = sm.OLS(m2['Part_D_Total_Premium'],m2[['Part_D_Drug_Deductible','unrestricted_drug','ivhaus']],missing='drop').fit()
print(model1s.summary())
with open('2. First_stage_regression_results.tex', 'w') as f:
    f.write(model1s.summary().as_latex())
#Second Stage
d=model1s.predict()
m2['predicted_Part_D_Total_Premium']= d
x2=m2.drop(['ivhaus','Part_D_Total_Premium','y'],axis=1) #drop ivhaus and premium 
model21 = sm.OLS(m2.y,x2, missing='drop').fit() 
print(model21.summary())

with open('2. Second_stage_regression_results.tex', 'w') as f:
    f.write(model21.summary().as_latex())


                                  OLS Regression Results                                 
Dep. Variable:     Part_D_Total_Premium   R-squared (uncentered):                   0.936
Model:                              OLS   Adj. R-squared (uncentered):              0.936
Method:                   Least Squares   F-statistic:                          4.311e+04
Date:                  Sun, 22 Dec 2024   Prob (F-statistic):                        0.00
Time:                          16:49:19   Log-Likelihood:                         -34785.
No. Observations:                  8795   AIC:                                  6.958e+04
Df Residuals:                      8792   BIC:                                  6.960e+04
Df Model:                             3                                                  
Covariance Type:              nonrobust                                                  
                             coef    std err          t      P>|t|      [0.025      0.975]
---------

In [49]:
#Statsmodel package function:
exo=m2.drop(['ivhaus','y','basic','predicted_Part_D_Total_Premium'],axis=1)
end=m2[['const','ivhaus','Part_D_Drug_Deductible', 'unrestricted_drug', 'in_area_flag']]
model22=IV2SLS(m2.y,exo,end).fit()
print(model22.summary())

                          IV2SLS Regression Results                           
Dep. Variable:                      y   R-squared:                       0.101
Model:                         IV2SLS   Adj. R-squared:                  0.100
Method:                     Two Stage   F-statistic:                     214.7
                        Least Squares   Prob (F-statistic):          4.43e-176
Date:                Sun, 22 Dec 2024                                         
Time:                        16:54:08                                         
No. Observations:                8795                                         
Df Residuals:                    8790                                         
Df Model:                           4                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     -2